# Limpieza de Datos — Yellow Taxi NYC

En este notebook se aplicarán reglas de limpieza y transformación a los datos de Yellow Taxi correspondientes a julio de 2026.

El objetivo es generar un dataset analítico limpio y reproducible, manteniendo los datos originales sin modificaciones.

Las decisiones de limpieza se basan en los hallazgos obtenidos durante la auditoría de calidad realizada en `01_data_audit.ipynb`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

ruta_raw = Path("../data/raw/yellow_tripdata_2026-07.parquet")
ruta_processed = Path("../data/processed/yellow_tripdata_2026-07_clean.parquet")

print("Archivo encontrado:", ruta_raw.exists())

Archivo encontrado: True


In [2]:
# Configurar formato numérico para facilitar la lectura
pd.options.display.float_format = "{:,.2f}".format

## Estrategia de limpieza

La limpieza se realizará de forma conservadora para evitar eliminar información válida sin evidencia suficiente.

### Reglas iniciales

1. **Conservar los datos originales**
   - El archivo ubicado en `data/raw/` no será modificado.

2. **Valores nulos**
   - No se eliminarán filas únicamente por contener valores nulos.
   - Los nulos estructurales detectados durante la auditoría se conservarán inicialmente.

3. **Duplicados**
   - No se eliminarán los 13.545 pares de registros similares, ya que no se detectaron duplicados exactos en las 21 variables analizadas.

4. **Distancia**
   - Los registros con `trip_distance = 0` no serán eliminados automáticamente.
   - Las distancias extremadamente altas serán identificadas como anomalías antes de decidir su tratamiento.

5. **Duración**
   - Se calculará la duración del viaje utilizando las fechas de inicio y finalización.
   - Se identificarán duraciones iguales a cero, negativas o extremadamente elevadas.

6. **Trazabilidad**
   - Las reglas aplicadas quedarán documentadas para que el proceso pueda reproducirse.

In [3]:
# Preparamos el archivo Parquet para procesarlo por bloques
parquet_file = pq.ParquetFile(ruta_raw)

print("Filas:", parquet_file.metadata.num_rows)
print("Columnas:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.metadata.num_row_groups)

Filas: 3530109
Columnas: 21
Row groups: 4


In [4]:
# Row Group 0
df = parquet_file.read_row_group(0).to_pandas()

df.shape

(1048576, 21)

In [5]:
# Creamos una variable nueva: duración del viaje en minutos.
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"]
    - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration_min"].describe()

count   1,048,576.00
mean           15.96
std            26.90
min             0.00
25%             7.13
50%            12.13
75%            20.07
max         5,678.68
Name: trip_duration_min, dtype: float64

In [6]:
# Creamos una flag que marca si un registro cumple una condición sospechosa.
df["flag_distance_zero"] = df["trip_distance"] == 0
df["flag_distance_extreme"] = df["trip_distance"] > 100

df["flag_duration_zero"] = df["trip_duration_min"] == 0
df["flag_duration_negative"] = df["trip_duration_min"] < 0
df["flag_duration_extreme"] = df["trip_duration_min"] > 180

In [7]:
# Contar los registros identificados por cada regla de calidad
columnas_flags = [
    "flag_distance_zero",
    "flag_distance_extreme",
    "flag_duration_zero",
    "flag_duration_negative",
    "flag_duration_extreme"
]

df[columnas_flags].sum()


flag_distance_zero        15460
flag_distance_extreme        21
flag_duration_zero        16752
flag_duration_negative        0
flag_duration_extreme       414
dtype: int64

In [8]:
# Crear una flag general para identificar registros con al menos una anomalía
df["flag_quality_issue"] = df[columnas_flags].any(axis=1)

df["flag_quality_issue"].value_counts()

flag_quality_issue
False    1016436
True       32140
Name: count, dtype: int64

In [9]:
# Definir una función reutilizable para aplicar las reglas de limpieza a cada bloque
def limpiar_bloque(df):
    df = df.copy()

    # Normalizar tipos entre Row Groups
    df["passenger_count"] = df["passenger_count"].astype("Int64")
    df["RatecodeID"] = df["RatecodeID"].astype("Int64")

    df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype("string")
    df["request_source"] = df["request_source"].astype("string")

    # Duración del viaje en minutos
    df["trip_duration_min"] = (
        df["tpep_dropoff_datetime"]
        - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    # Flags de calidad
    df["flag_distance_zero"] = df["trip_distance"] == 0
    df["flag_distance_extreme"] = df["trip_distance"] > 100

    df["flag_duration_zero"] = df["trip_duration_min"] == 0
    df["flag_duration_negative"] = df["trip_duration_min"] < 0
    df["flag_duration_extreme"] = df["trip_duration_min"] > 180

    # Flag general
    columnas_flags = [
        "flag_distance_zero",
        "flag_distance_extreme",
        "flag_duration_zero",
        "flag_duration_negative",
        "flag_duration_extreme"
    ]

    df["flag_quality_issue"] = df[columnas_flags].any(axis=1)

    return df

In [10]:
# Aplicar la función de limpieza al primer bloque
df_limpio = limpiar_bloque(df)

In [11]:
# Verificar las dimensiones del bloque después de aplicar la limpieza
df_limpio.shape

(1048576, 28)

In [12]:
# Revisar las variables de calidad generadas después de la limpieza
df_limpio[
    [
        "trip_duration_min",
        "flag_distance_zero",
        "flag_distance_extreme",
        "flag_duration_zero",
        "flag_duration_negative",
        "flag_duration_extreme",
        "flag_quality_issue"
    ]
].head()

,trip_duration_min,flag_distance_zero,flag_distance_extreme,flag_duration_zero,flag_duration_negative,flag_duration_extreme,flag_quality_issue
0,6.88,False,False,False,False,False,False
1,14.32,False,False,False,False,False,False
2,8.70,False,False,False,False,False,False
3,22.75,False,False,False,False,False,False
4,0.00,False,False,True,False,False,True


In [13]:
# Liberar memoria antes de procesar el dataset completo
del df
del df_limpio

import gc
gc.collect()

71

In [14]:
import pyarrow as pa

In [15]:
# Automatizar la limpieza de todos los bloques
writer = None

try:
    for i in range(parquet_file.metadata.num_row_groups):

        print(
            f"Procesando Row Group {i + 1}/"
            f"{parquet_file.metadata.num_row_groups}..."
        )

        # Leer un solo bloque
        bloque = parquet_file.read_row_group(i).to_pandas()

        # Aplicar la función de limpieza
        bloque_limpio = limpiar_bloque(bloque)

        # Convertir a tabla PyArrow
        tabla = pa.Table.from_pandas(
            bloque_limpio,
            preserve_index=False
        )

        # Crear el archivo con el primer bloque
        if writer is None:
            writer = pq.ParquetWriter(
                ruta_processed,
                tabla.schema,
                compression="snappy"
            )

        # Agregar el bloque al archivo
        writer.write_table(tabla)

        # Liberar memoria
        del bloque
        del bloque_limpio
        del tabla

        gc.collect()

        print(f"Row Group {i + 1} completado.")

finally:
    if writer is not None:
        writer.close()

print("Procesamiento terminado.")

Procesando Row Group 1/4...
Row Group 1 completado.
Procesando Row Group 2/4...
Row Group 2 completado.
Procesando Row Group 3/4...
Row Group 3 completado.
Procesando Row Group 4/4...
Row Group 4 completado.
Procesamiento terminado.


In [16]:
# Validamos el resultado
archivo_limpio = pq.ParquetFile(ruta_processed)

print("Filas:", archivo_limpio.metadata.num_rows)
print("Columnas:", archivo_limpio.metadata.num_columns)
print("Row Groups:", archivo_limpio.metadata.num_row_groups)
print("Archivo existe:", ruta_processed.exists())
print(
    "Tamaño MB:",
    round(ruta_processed.stat().st_size / (1024 ** 2), 2)
)

Filas: 3530109
Columnas: 28
Row Groups: 4
Archivo existe: True
Tamaño MB: 76.59


In [17]:
# Validar globalmente la cantidad de registros identificados por cada flag
flags_validacion = [
    "flag_distance_zero",
    "flag_distance_extreme",
    "flag_duration_zero",
    "flag_duration_negative",
    "flag_duration_extreme",
    "flag_quality_issue"
]

totales_flags = {flag: 0 for flag in flags_validacion}

for i in range(archivo_limpio.metadata.num_row_groups):

    bloque_flags = archivo_limpio.read_row_group(
        i,
        columns=flags_validacion
    ).to_pandas()

    for flag in flags_validacion:
        totales_flags[flag] += bloque_flags[flag].sum()

    del bloque_flags
    gc.collect()

pd.Series(totales_flags)

flag_distance_zero        128322
flag_distance_extreme        172
flag_duration_zero         42315
flag_duration_negative         1
flag_duration_extreme       1002
flag_quality_issue        170490
dtype: int64

## Resultado del proceso de limpieza

Se procesaron **3.530.109 registros** mediante un pipeline por bloques para reducir el consumo de memoria.

El dataset original de 21 variables se transformó en un dataset analítico de **28 variables**, conservando todos los registros originales y agregando variables de duración y control de calidad.

### Resultados de las reglas de calidad

| Control de calidad | Registros |
|---|---:|
| Distancia igual a 0 | 128.322 |
| Distancia superior a 100 millas | 172 |
| Duración igual a 0 | 42.315 |
| Duración negativa | 1 |
| Duración superior a 180 minutos | 1.002 |
| Registros con al menos una anomalía | 170.490 |

Aproximadamente el **4,83 %** de los registros activa al menos una regla de calidad.

Los registros identificados no fueron eliminados. Se conservaron mediante indicadores (`flags`) para permitir diferentes criterios de filtrado según el análisis posterior.

El procesamiento se realizó por Row Groups para evitar cargar el dataset completo en memoria.

### Salida generada

`data/processed/yellow_tripdata_2026-07_clean.parquet`

- Registros: **3.530.109**
- Variables: **28**
- Row Groups: **4**
- Tamaño aproximado: **76,59 MB**

El archivo original permanece sin modificaciones en `data/raw/`.